# Atividade 01 - Análise Exploratória e Avaliação de Qualidade dos Dados
**Disciplina:** Sistemas Inteligentes Orientados a Dados (PGCOMP / UFBA - 2026.2)
**Professor:** Maycon Leone Peixoto 



In [1]:
import re
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações visuais e de exibição do Pandas
sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## 1. Carregamento dos Dados


In [5]:
dados = Path('../dados/Synthetic_Financial_datasets_log.csv')

def carregar_dados(caminho: Path) -> pd.DataFrame:
    """Carrega o dataset CSV e retorna um DataFrame do Pandas."""
    if not caminho.is_file():
        print(f"[ERRO] O arquivo '{caminho}' não foi encontrado. Verifique o diretório.")
        return pd.DataFrame()
    return pd.read_csv(caminho)

df = carregar_dados(dados)


## 2. Caracterização Geral da Base


In [6]:
if not df.empty:
    print("--- Informações Estruturais ---")
    df.info()
    
    print("\n--- Amostra Inicial (Head) ---")
    display(df.head())

--- Informações Estruturais ---
<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB

--- Amostra Inicial (Head) ---


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


## 3.  Qualidade de Dados


In [8]:
if not df.empty:
    print("\n--- Valores Ausentes ---")
    null_counts = df.isnull().sum()
    null_pct = df.isnull().mean() * 100
    df_nulos = pd.DataFrame({'Total Ausentes': null_counts, 'Percentual (%)': null_pct})
    display(df_nulos[df_nulos['Total Ausentes'] > 0])
    
    print("\n--- Duplicidades ---")
    dup_total = df.duplicated().sum()
    dup_id = df.duplicated(subset=['id_transacao']).sum() if 'id_transacao' in df.columns else 0
    print(f"Linhas idênticas: {dup_total}")
    print(f"IDs de transação duplicados: {dup_id}")


--- Valores Ausentes ---


,Total Ausentes,Percentual (%)



--- Duplicidades ---
Linhas idênticas: 0
IDs de transação duplicados: 0


### 3.1. Validação de Formatos e Outliers


In [9]:
if not df.empty and 'data_transacao' in df.columns:
    print("--- Avaliação de Formato de Datas ---")
    datas_conversao = pd.to_datetime(df['data_transacao'], errors='coerce')
    datas_invalidas = df['data_transacao'].notnull() & datas_conversao.isnull()
    n_datas_invalidas = datas_invalidas.sum()
    
    print(f"Datas não convertíveis (formatos inválidos): {n_datas_invalidas}")
    if n_datas_invalidas > 0:
        display(df[datas_invalidas]['data_transacao'].head())

if not df.empty and 'valor' in df.columns:
    print("\n--- Outliers Extremos de Valor ---")
    outliers_valores = df[(df['valor'] > 50000) | (df['valor'] < -1000)]
    print(f"Transações suspeitas (> R$ 50k ou < R$ -1k): {len(outliers_valores)}")
    if not outliers_valores.empty:
        display(outliers_valores[['id_transacao', 'descricao_original', 'valor']].head())

## 4. Tratamento e Limpeza 


In [10]:
def limpar_texto(texto: str) -> str:

    if not isinstance(texto, str):
        return ""
    texto = texto.upper()
    texto = re.sub(r'PG \\*', '', texto)       # Remove marcas de gateway
    texto = re.sub(r' BR$', '', texto)         # Remove sufixo de país
    texto = re.sub(r'- REF \\d+', '', texto)   # Remove códigos de referência randômicos
    texto = re.sub(r'#\\w+', '', texto)        # Remove hashtags de sistema
    texto = re.sub(r'\\s+', ' ', texto).strip() # Remove múltiplos espaços
    return texto

if not df.empty:
    # Uso de .copy() evita o erro de SettingWithCopyWarning do Pandas
    df_limpo = df.drop_duplicates(subset=['id_transacao']).copy() if 'id_transacao' in df.columns else df.copy()
    
    if 'data_transacao' in df_limpo.columns:
        df_limpo['data_datetime'] = pd.to_datetime(df_limpo['data_transacao'], errors='coerce')
        df_limpo['data_datetime'] = df_limpo['data_datetime'].ffill()
    
    if 'descricao_original' in df_limpo.columns:
        df_limpo['descricao_limpa'] = df_limpo['descricao_original'].apply(limpar_texto)
    


## 5. Análise de Distribuição e Exportação


In [12]:
if not df.empty and 'tipo_conta_estimado' in df_limpo.columns:
    dist_target = df_limpo['tipo_conta_estimado'].value_counts()
    dist_pct = df_limpo['tipo_conta_estimado'].value_counts(normalize=True) * 100
    
    df_dist = pd.DataFrame({'Contagem': dist_target, 'Proporção (%)': dist_pct.round(2)})
    print("--- Distribuição da Variável-Alvo ---")
    display(df_dist)
    
    plt.figure(figsize=(8, 4))
    sns.countplot(data=df_limpo, x='tipo_conta_estimado', palette='mako', hue='tipo_conta_estimado')
    plt.title('Proporção do Tipo de Conta Estimado')
    plt.xlabel('Tipo de Conta')
    plt.ylabel('Volume de Transações')
    plt.tight_layout()
    plt.show()